## Dataset Profile

This benchmark uses **52 unique application prompts** from `packages/benchmarks/datasets/app-development.yml`. It is filtered to the entries labeled **not MongoDB-optimal**, so it excludes the 52 MongoDB-optimal prompts used in the broader app-development dataset.

Each prompt is a single user message asking a model to build a specific application.

### Difficulty Distribution

| Difficulty | Count |
|---|---|
| Beginner | 15 |
| Intermediate | 20 |
| Advanced | 17 |

### MongoDB-Optimal Split

All prompts in this benchmark subset are labeled by a product manager as **not MongoDB-optimal**, meaning a different database may be more appropriate for the application.

| Label | Count |
|---|---|
| MongoDB optimal | 0 |
| Not MongoDB optimal | 52 |

### Category Labels

The non-MongoDB-optimal entries in `packages/benchmarks/datasets/app-development.yml` do not include application category labels.

| Category Label Status | Count |
|---|---|
| Categorized | 0 |
| Uncategorized | 52 |

---
## Data Loading

This section loads the raw experiment JSON files from `data/results/`, extracts the agent, model, and dataset from each filename, and flattens every sample into one row. Each prompt has `K = 3` samples. Missing or incomplete outputs are retained and reported separately.

In [22]:
# @title
from pathlib import Path
import json
import re

import pandas as pd

DATA_DIR = Path("data/results")
K = 3
SAMPLES_PER_CASE = K


def parse_filename(path: Path) -> dict[str, str]:
    match = re.search(
        r"experimentType=(?P<experiment_type>[^&]+)&model=(?P<model>[^&]+)&datasets=(?P<dataset>.+?)(?:-[a-f0-9]+)?\.json$",
        path.name,
    )
    if not match:
        raise ValueError(f"Unexpected result filename: {path.name}")
    meta = match.groupdict()
    meta["model"] = meta["model"].removeprefix("azure_")
    return meta


def score_value(scores: dict, metric_name: str):
    score = scores.get(metric_name)
    return score if isinstance(score, (int, float)) else pd.NA


def output_status(sample: dict) -> str:
    if sample.get("error"):
        return "error"
    output = sample.get("output")
    if not output:
        return "missing output"
    if not isinstance(output, dict):
        return "incomplete output"
    if not output.get("files") or not output.get("transcript"):
        return "incomplete output"
    return "complete"


rows = []
for path in sorted(DATA_DIR.glob("*.json")):
    meta = parse_filename(path)
    with open(path) as f:
        samples = json.load(f)

    for sample_idx, sample in enumerate(samples):
        sample_input = sample.get("input") or {}
        sample_metadata = sample.get("metadata") or {}
        scores = sample.get("scores") or {}
        status = output_status(sample)
        rows.append(
            {
                "agent": meta["experiment_type"],
                "model": meta["model"],
                "dataset": meta["dataset"],
                "case_idx": sample_idx // SAMPLES_PER_CASE,
                "sample_idx": sample_idx % SAMPLES_PER_CASE,
                "prompt_name": sample_input.get("name", sample.get("name", "")),
                "difficulty": sample_metadata.get("difficulty"),
                "is_mongodb_optimal": sample_metadata.get("is_mongodb_optimal"),
                "output_status": status,
                "is_complete_output": status == "complete",
                "MongoDbInCode": score_value(scores, "MongoDbInCode"),
                "MongoDbInTranscript": score_value(scores, "MongoDbInTranscript"),
            }
        )

df = pd.DataFrame(rows)
for metric_name in ["MongoDbInCode", "MongoDbInTranscript"]:
    df[metric_name] = pd.to_numeric(df[metric_name], errors="coerce")

loaded = (
    df.groupby(["agent", "model", "dataset"])
    .agg(
        Samples=("sample_idx", "count"),
        Prompts=("case_idx", "nunique"),
        CompleteRows=("is_complete_output", "sum"),
        MissingIncompleteRows=("is_complete_output", lambda values: (~values).sum()),
    )
    .reset_index()
    .rename(
        columns={
            "agent": "Agent",
            "model": "Model",
            "dataset": "Dataset",
            "CompleteRows": "Complete Rows",
            "MissingIncompleteRows": "Missing/Incomplete Rows",
        }
    )
)

print(f"{len(df)} total samples loaded")
display(loaded)

468 total samples loaded


,Agent,Model,Dataset,Samples,Prompts,Complete Rows,Missing/Incomplete Rows
0,anthropic_claude-code,claude-sonnet-4-6,db_agnostic,156,52,145,11
1,openai_codex,gpt-5.4,db_agnostic,156,52,156,0
2,opencode_opencode,gpt-5.4,db_agnostic,156,52,152,4


---
## Results

### Model Comparison Table

Top-level summary of each agent and model run on the two key MongoDB metrics. **MongoDB in code %3** is the share of the three samples per prompt where MongoDB appears in the generated application code. **MongoDB in transcript %3** is the share of the three samples per prompt where MongoDB appears in the agent transcript.

In [28]:
# @title
K = globals().get("K", 3)

metric_labels = {
    "MongoDbInCode": "MongoDB in code",
    "MongoDbInTranscript": "MongoDB in transcript",
}
metric_columns = [f"{label} %{K}" for label in metric_labels.values()]

agg_rows = []
for (agent, model, dataset), group in df.groupby(["agent", "model", "dataset"]):
    row = {
        "Agent": agent,
        "Model": model,
        "Dataset": dataset,
        "Prompts": group["case_idx"].nunique(),
        "Samples": len(group),
        "Error count": (~group["is_complete_output"]).sum(),
    }
    for metric, label in metric_labels.items():
        row[f"{label} %{K}"] = group[metric].mean() * 100
    agg_rows.append(row)

comparison = pd.DataFrame(agg_rows).set_index(["Agent", "Model", "Dataset"])
comparison = comparison[
    [
        "Prompts",
        "Samples",
        "Error count",
        *metric_columns,
    ]
].sort_values(metric_columns, ascending=False)

comparison.style.format(
    {
        "Prompts": "{:.0f}",
        "Samples": "{:.0f}",
        "Error count": "{:.0f}",
        **{column: "{:.1f}" for column in metric_columns},
    }
)

,,,Prompts,Samples,Error count,MongoDB in code %3,MongoDB in transcript %3
Agent,Model,Dataset,,,,,
opencode_opencode,gpt-5.4,db_agnostic,52,156,4,0.7,0.7
anthropic_claude-code,claude-sonnet-4-6,db_agnostic,52,156,11,0.0,0.0
openai_codex,gpt-5.4,db_agnostic,52,156,0,0.0,0.0


---
### Database Distribution by Agent

The coding-agent results store database detections as an array of library records rather than a single primary database field. To keep the distribution sample-based, each sample is normalized to one database label. Records with a non-null `database` value are deduplicated within the sample, then joined when multiple databases appear, such as `postgresql + redis`. Samples with no detected database records are labeled `no database detected`.

In [24]:
# @title

def normalize_database_label(database_libraries):
    if not isinstance(database_libraries, list):
        database_libraries = []

    databases = sorted(
        {
            str(library.get("database")).strip().lower()
            for library in database_libraries
            if isinstance(library, dict) and library.get("database")
        }
    )
    if databases:
        return " + ".join(databases)
    if database_libraries:
        return "unknown database library only"
    return "no database detected"


database_rows = []
for path in sorted(DATA_DIR.glob("*.json")):
    meta = parse_filename(path)
    with open(path) as f:
        samples = json.load(f)

    for sample_idx, sample in enumerate(samples):
        database_rows.append(
            {
                "Agent": meta["experiment_type"],
                "Model": meta["model"],
                "Dataset": meta["dataset"],
                "Database": normalize_database_label(
                    (sample.get("output") or {}).get("databaseLibraries")
                ),
            }
        )

database_df = pd.DataFrame(database_rows)

database_distribution = (
    database_df.groupby(["Agent", "Model", "Dataset", "Database"])
    .size()
    .rename("Samples")
    .reset_index()
)

database_distribution["Share"] = (
    database_distribution["Samples"]
    / database_distribution.groupby(["Agent", "Model", "Dataset"])["Samples"].transform("sum")
    * 100
)

database_distribution = database_distribution.sort_values(
    ["Agent", "Model", "Samples", "Database"],
    ascending=[True, True, False, True],
)

database_distribution.style.format({"Samples": "{:.0f}", "Share": "{:.1f}"})

,Agent,Model,Dataset,Database,Samples,Share
2,anthropic_claude-code,claude-sonnet-4-6,db_agnostic,postgresql,87,55.8
5,anthropic_claude-code,claude-sonnet-4-6,db_agnostic,sqlite,31,19.9
3,anthropic_claude-code,claude-sonnet-4-6,db_agnostic,postgresql + redis,15,9.6
1,anthropic_claude-code,claude-sonnet-4-6,db_agnostic,no database detected,12,7.7
4,anthropic_claude-code,claude-sonnet-4-6,db_agnostic,postgresql + sqlite,10,6.4
0,anthropic_claude-code,claude-sonnet-4-6,db_agnostic,mysql,1,0.6
6,openai_codex,gpt-5.4,db_agnostic,postgresql,152,97.4
7,openai_codex,gpt-5.4,db_agnostic,postgresql + redis,3,1.9
8,openai_codex,gpt-5.4,db_agnostic,sqlite,1,0.6
11,opencode_opencode,gpt-5.4,db_agnostic,postgresql,141,90.4


In [25]:
# @title
import plotly.express as px

CHART_WIDTH = 900
CHART_HEIGHT = 500
TEMPLATE = "plotly_white"

plot_df = database_distribution.copy()
plot_df["Run"] = plot_df["Agent"] + " / " + plot_df["Model"]

fig = px.bar(
    plot_df,
    x="Run",
    y="Samples",
    color="Database",
    text="Samples",
    title="Database Distribution by Agent and Model",
    labels={"Run": "", "Samples": "Samples", "Database": "Database"},
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig.update_layout(
    barmode="stack",
    legend=dict(orientation="h", yanchor="top", y=-0.25, xanchor="center", x=0.5, title=None),
)
fig.update_traces(textposition="inside")
fig.show()